# Analysis of Drinking Waterpoints in Lagos, Nigeria
> Note: This notebook requires the local environment dependencies listed in our [requirements.txt] (requirements.txt) file. Use this file to install the required packages in a virtual environment.

> To excecute OpenRouteService functions, it is required to install the [library dependencies](https://github.com/GIScience/openrouteservice-examples#local-installation). You should either have an [openrouteservice API key](https://openrouteservice.org/dev/#/signup) or a local ORS environment to complete the analysis.

The model concepts and processes are described in our documentation. The [Dataset-interpretability](https://github.com/urbanbigdatacentre/ideamaps-models/blob/a4084fb650424ac575941cdacb71421aa882bae4/models/emergency-maternal-care/kano/dataset-interpretability.md) file describes the rationale behind this model.

## Workflow:
The notebook is divided into the following sections:

1. Initial Setup
2. Data Preparation
3. Travel time estimates
4. Two-step floating catchment area (2SFCA) analysis
5. Results

## 1. Initial Setup

## Setting up the virtual environment

```bash
# Create a new virtual environment
# It is recommended to create this virtual environment in the scripts folder
python -m venv .venv

# Activate the virtual environment
source .venv/bin/activate
pip install -r requirements.txt
```

## To run your notebook in VS Code

```bash
pip install -U ipykernel
python -m ipykernel install --user --name=.venv
```

In [1]:
import geopandas as gpd
import os
import numpy as np
import pandas as pd

import openrouteservice
from dotenv import load_dotenv

import rasterio
from rasterio.mask import mask

from shapely.geometry import Point

from pathlib import Path
from shapely.geometry import Polygon

import requests
import math
from math import *
from sklearn.preprocessing import MinMaxScaler
from shapely.geometry import box
from scipy.spatial import cKDTree

## Preprocessing
In this study, users first requested an ORS Matrix API key from the [OpenRouteService](https://openrouteservice.org/) platform and subsequently interacted with the OpenRouteService API through the instantiation of the OpenRouteService client. This is the OpenRouteService [API documentation](https://openrouteservice.org/dev/#/api-docs/introduction) for ORS Core-Version 9.0.0. 

Generate a [API Key](https://openrouteservice.org/dev/#/home?tab=1) (Token) it is necessary to sign up at the OpenRouteService dashboard by using your E-mail address or sign up with your GitHub. After logging in, go to the Dashboard by clicking on your profile icon and navigate to the API Keys section. Click "Create API Key" to generate a free key and then choose a service plan (the free plan has limited requests per day). Copy the API Key and store it securely. 

OpenRouteService primarily uses API keys for authentication. However, if a token is required for certain endpoints, you can send a request with your API key in the Authorization header. This process facilitated various geospatial analysis functions, including isochrone generation.

### Option 1: Using an ORS API Key
Make sure you have a .env file in the root directory with the following content:
```bash
    OPENROUTESERVICE_API_KEY='your_api_key'
```

In [ ]:
# %%
# Read the api key from the .env file
from dotenv import load_dotenv
%load_ext dotenv
%dotenv
api_key = os.getenv('OPENROUTESERVICE_API_KEY')
client = openrouteservice.Client(key=api_key)

### Option 2: Using a local ORS service
Make sure you have set a local service that runs the OSM-based ORS API. 
```r
    # Insert R code from the local ORS service
```

For the drinking‑water model in Lagos State, we began with 485 waterpoint records from the [Donate Water initiative](https://donatewater.ng), a community‑driven platform that combines crowdsourcing and data analytics to map and monitor water facilities across Nigeria. In [QGIS](https://qgis.org/), to be continued.

Boundary Framework
When these 485 non‑outlier points were overlaid on the official Lagos Local Government Area (Admin Level 2) boundary [Humanitarian Data Exchange, published 25 Nov 2015](https://data.humdata.org/) , tbd.

Usage in Analysis
From that point onward, every spatial query, point‑density calculation, and service‑area model was executed strictly within our hand‑digitized boundary. By anchoring the analysis to the actual distribution of cleaned, functional waterpoints, we ensure that our density and coverage metrics faithfully represent on‑the‑ground service provision—avoiding the underestimation that would result from forcing the data into administrative lines misaligned with real‑world facility locations.

Data Sources
* [Waterpoint facilities](https://donatewater.ng): generated 12 Oct 2023; accessed 15 Feb 2025 
* [District boundaries](https://data.humdata.org/dataset/nigeria-admin-level-2): Humanitarian Data Exchange, published 25 Nov 2015; accessed 10 Jan 2024
* [World Population Data](https://hub.worldpop.org/geodata/summary?id=28031): generated 2020-02-01; accessed 15 july 2025

In [2]:
data_inputs = '../Lagos/data-inputs/'                # Current folder
data_temp = '../Lagos/data-temp/'       # Goes up one folder, then into data-temp
model_outputs = '../Lagos/model-output/'            # Goes up one folder to scripts/Lagos

## Data Collection

### 1.1 Drinking waterpoints for Kano

In [38]:
# Read the drinking water points data
# Ensure the file path is correct and the file exists
drinking_waterpoints = gpd.read_file(data_inputs + 'Lagos_DW_Raw.gpkg')

In [39]:
drinking_waterpoints

,Id,ImageUrls,ImageUrls.,Status,Status.key,Submission,UserEmail,UserEmail.,_id,_ignored,...,globalid,uniq_id,timestamp,editor,statename,statecode,capcity,source,geozone,geometry
0,"1,615",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:02:51.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,SHQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,16fa258c-be83-44c4-836e-d4768129a329,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.3942 6.50181)
1,"1,606",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 11:26:32.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,R3Q0CJABf4CpNA64g0Fo,-,...,16fa258c-be83-44c4-836e-d4768129a329,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39393 6.49864)
2,"1,605",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 11:21:21.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,RnQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,16fa258c-be83-44c4-836e-d4768129a329,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39384 6.49949)
3,"1,617",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:07:24.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,RHQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,16fa258c-be83-44c4-836e-d4768129a329,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39375 6.50247)
4,"1,607",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 11:31:18.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,THQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,16fa258c-be83-44c4-836e-d4768129a329,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39521 6.49856)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279,"1,594",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 10:49:33.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,K3Q0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,16fa258c-be83-44c4-836e-d4768129a329,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39158 6.50154)
280,"1,624",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:51:47.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,MnQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,16fa258c-be83-44c4-836e-d4768129a329,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39231 6.50446)
281,"1,596",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 10:58:00.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,MXQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,16fa258c-be83-44c4-836e-d4768129a329,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39204 6.50114)
282,"1,623",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:39:30.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,MHQ0CJABf4CpNA64g0Fo,-,...,16fa258c-be83-44c4-836e-d4768129a329,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39194 6.50362)


In [40]:
# Define unwanted categories
unwanted_sources = ["No, it is not functional and not in use", "No, it is abandoned", "I don't know"]

# Filter them out
drinking_waterpoints = drinking_waterpoints[~drinking_waterpoints['waterSourc'].isin(unwanted_sources)]


In [41]:
# Map the water source valuess to better descriptive labels
# Ensure the mapping is correct and matches the data
drinking_waterpoints['waterSourc'] = (
    drinking_waterpoints['waterSourc']
    .astype(str)
    .str.strip()
    .map({
        'Yes, it is fully functional and still in use': 'Fully Functional',
        'Yes, it is partially functional and still in use': 'Partially Functional'
    })
)

/opt/anaconda3/lib/python3.12/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [42]:
# Define unwanted categories
unwanted_sources1 = ["Don't know"]

# Filter them out
drinking_waterpoints = drinking_waterpoints[~drinking_waterpoints['waterPubli'].isin(unwanted_sources1)]
drinking_waterpoints = drinking_waterpoints[~drinking_waterpoints['waterFree'].isin(unwanted_sources1)]


In [43]:
drinking_waterpoints

,Id,ImageUrls,ImageUrls.,Status,Status.key,Submission,UserEmail,UserEmail.,_id,_ignored,...,globalid,uniq_id,timestamp,editor,statename,statecode,capcity,source,geozone,geometry
0,"1,615",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:02:51.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,SHQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,16fa258c-be83-44c4-836e-d4768129a329,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.3942 6.50181)
2,"1,605",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 11:21:21.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,RnQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,16fa258c-be83-44c4-836e-d4768129a329,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39384 6.49949)
3,"1,617",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:07:24.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,RHQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,16fa258c-be83-44c4-836e-d4768129a329,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39375 6.50247)
4,"1,607",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 11:31:18.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,THQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,16fa258c-be83-44c4-836e-d4768129a329,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39521 6.49856)
5,"1,612",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 11:55:40.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,S3Q0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,16fa258c-be83-44c4-836e-d4768129a329,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39507 6.49923)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279,"1,594",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 10:49:33.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,K3Q0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,16fa258c-be83-44c4-836e-d4768129a329,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39158 6.50154)
280,"1,624",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:51:47.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,MnQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,16fa258c-be83-44c4-836e-d4768129a329,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39231 6.50446)
281,"1,596",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 10:58:00.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,MXQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,16fa258c-be83-44c4-836e-d4768129a329,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39204 6.50114)
282,"1,623",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:39:30.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,MHQ0CJABf4CpNA64g0Fo,-,...,16fa258c-be83-44c4-836e-d4768129a329,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39194 6.50362)


In [44]:
# Define mapping from waterState to waterDrink for "I don't know"
state_to_drink = {
    "Clear": "Yes",
    "Dirty": "No",
    "Currently no water available": "No",
    "Muddy": "No",
    "Coloured": "No"
}

# Copy the original waterDrink column
drinking_waterpoints['waterDrink_estimated'] = drinking_waterpoints['waterDrink']

# Only update values in waterDrink_estimated where waterDrink is "I don't know"
drinking_waterpoints.loc[drinking_waterpoints['waterDrink'] == "Don't know", 'waterDrink_estimated'] = drinking_waterpoints.loc[
    drinking_waterpoints['waterDrink'] == "Don't know", 'waterState'
].map(state_to_drink)

In [45]:
drinking_waterpoints

,Id,ImageUrls,ImageUrls.,Status,Status.key,Submission,UserEmail,UserEmail.,_id,_ignored,...,uniq_id,timestamp,editor,statename,statecode,capcity,source,geozone,geometry,waterDrink_estimated
0,"1,615",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:02:51.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,SHQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.3942 6.50181),Yes
2,"1,605",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 11:21:21.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,RnQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39384 6.49949),No
3,"1,617",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:07:24.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,RHQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39375 6.50247),Yes
4,"1,607",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 11:31:18.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,THQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39521 6.49856),No
5,"1,612",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 11:55:40.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,S3Q0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39507 6.49923),No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279,"1,594",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 10:49:33.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,K3Q0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39158 6.50154),No
280,"1,624",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:51:47.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,MnQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39231 6.50446),No
281,"1,596",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 10:58:00.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,MXQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39204 6.50114),No
282,"1,623",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:39:30.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,MHQ0CJABf4CpNA64g0Fo,-,...,1179,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39194 6.50362),No


In [46]:
# Define user-specified improved water source types
improved = [
    'piped-borne water',
    'borehole with hand pump',
    'borehole with tap',
    'water vendors',
    'spring'
]

# Create a new column 'watertype_revised' with classification
drinking_waterpoints['watertype_revised'] = drinking_waterpoints['waterType'].apply(
    lambda x: 'Improved' if isinstance(x, str) and x.strip().lower() in improved else 'Unimproved'
)

drinking_waterpoints

,Id,ImageUrls,ImageUrls.,Status,Status.key,Submission,UserEmail,UserEmail.,_id,_ignored,...,timestamp,editor,statename,statecode,capcity,source,geozone,geometry,waterDrink_estimated,watertype_revised
0,"1,615",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:02:51.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,SHQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.3942 6.50181),Yes,Unimproved
2,"1,605",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 11:21:21.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,RnQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39384 6.49949),No,Unimproved
3,"1,617",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:07:24.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,RHQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39375 6.50247),Yes,Unimproved
4,"1,607",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 11:31:18.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,THQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39521 6.49856),No,Unimproved
5,"1,612",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 11:55:40.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,S3Q0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39507 6.49923),No,Unimproved
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279,"1,594",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 10:49:33.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,K3Q0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39158 6.50154),No,Unimproved
280,"1,624",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:51:47.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,MnQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39231 6.50446),No,Improved
281,"1,596",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 10:58:00.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,MXQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39204 6.50114),No,Unimproved
282,"1,623",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:39:30.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,MHQ0CJABf4CpNA64g0Fo,-,...,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39194 6.50362),No,Improved


In [47]:
# map the waterDrink_estimated values to more descriptive labels
# Ensure the mapping is correct and matches the data
drinking_waterpoints['waterDrink_estimated'] = (
    drinking_waterpoints['waterDrink_estimated']
    .astype(str)
    .str.strip()
    .map({
        'Yes': 'Drinkable',
        'No': 'Non drinkable'
    })
)

In [48]:
# map the waterFree values to more descriptive labels
# Ensure the mapping is correct and matches the data
drinking_waterpoints['waterFree'] = (
    drinking_waterpoints['waterFree']
    .astype(str)
    .str.strip()
    .map({
        'Yes': 'Free',
        'No': 'Non Free'
    })
)

In [49]:
drinking_waterpoints

,Id,ImageUrls,ImageUrls.,Status,Status.key,Submission,UserEmail,UserEmail.,_id,_ignored,...,timestamp,editor,statename,statecode,capcity,source,geozone,geometry,waterDrink_estimated,watertype_revised
0,"1,615",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:02:51.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,SHQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.3942 6.50181),Drinkable,Unimproved
2,"1,605",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 11:21:21.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,RnQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39384 6.49949),Non drinkable,Unimproved
3,"1,617",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:07:24.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,RHQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39375 6.50247),Drinkable,Unimproved
4,"1,607",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 11:31:18.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,THQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39521 6.49856),Non drinkable,Unimproved
5,"1,612",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 11:55:40.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,S3Q0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39507 6.49923),Non drinkable,Unimproved
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279,"1,594",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 10:49:33.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,K3Q0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39158 6.50154),Non drinkable,Unimproved
280,"1,624",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:51:47.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,MnQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39231 6.50446),Non drinkable,Improved
281,"1,596",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 10:58:00.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,MXQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39204 6.50114),Non drinkable,Unimproved
282,"1,623",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:39:30.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,MHQ0CJABf4CpNA64g0Fo,-,...,2019-07-17,nuraddeen.isah,Lagos,LA,Lagos,eHA_Polio,SWZ,POINT (3.39194 6.50362),Non drinkable,Improved


In [50]:
# 1. Load or reference your two GeoDataFrames
#    – drinking_waterpoints: your 456 waterpoints
#    – study_area: the polygon(s) defining your Kano study area
#      (e.g. from "grid-boundary-kano.gpkg")
study_area = gpd.read_file(data_inputs + "100mGrid.gpkg")

# 2. Ensure both are in the same CRS
if drinking_waterpoints.crs != study_area.crs:
    drinking_waterpoints = drinking_waterpoints.to_crs(study_area.crs)

drinking_waterpoints = gpd.sjoin(drinking_waterpoints, study_area, how="inner", predicate="intersects")
drinking_waterpoints = drinking_waterpoints[drinking_waterpoints.columns]
drinking_waterpoints

,Id,ImageUrls,ImageUrls.,Status,Status.key,Submission,UserEmail,UserEmail.,_id,_ignored,...,watertype_revised,index_right,id,rowid,latitude,lat_min,lat_max,longitude,lon_min,lon_max
0,"1,615",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:02:51.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,SHQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,Unimproved,197419,147427609.0,197420,6.501777,6.501371,6.502182,3.394455,3.393952,3.394957
2,"1,605",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 11:21:21.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,RnQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,Unimproved,196869,147410640.0,196870,6.499344,6.498939,6.499750,3.393443,3.392940,3.393946
3,"1,617",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:07:24.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,RHQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,Unimproved,196865,147410636.0,196866,6.502587,6.502182,6.502993,3.393457,3.392954,3.393959
4,"1,607",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 11:31:18.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,THQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,Unimproved,197977,147444585.0,197978,6.498534,6.498128,6.498939,3.395443,3.394941,3.395946
5,"1,612",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 11:55:40.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,S3Q0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,Unimproved,197976,147444584.0,197977,6.499344,6.498939,6.499750,3.395447,3.394944,3.395949
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279,"1,594",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 10:49:33.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,K3Q0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,Unimproved,195760,147376693.0,195761,6.501777,6.501371,6.502182,3.391450,3.390947,3.391952
280,"1,624",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:51:47.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,MnQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,Improved,196310,147393662.0,196311,6.504209,6.503804,6.504614,3.392462,3.391959,3.392964
281,"1,596",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 10:58:00.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,MXQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,Unimproved,196314,147393666.0,196315,6.500966,6.500561,6.501371,3.392448,3.391945,3.392951
282,"1,623",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:39:30.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,MHQ0CJABf4CpNA64g0Fo,-,...,Improved,195758,147376691.0,195759,6.503398,6.502993,6.503804,3.391456,3.390954,3.391959


In [51]:
# Define conditions for categorizing water points into limited, optimal, and moderate water categories
# Ensure the conditions are correctly defined based on the data
conditions = [
    drinking_waterpoints['waterDrink_estimated'] == 'Non drinkable',
    (drinking_waterpoints['waterSourc'] == 'Fully Functional') & (drinking_waterpoints['watertype_revised'] == 'Improved')
]

categories = ['Limited Water', 'Optimal Water']

drinking_waterpoints['category'] = np.select(conditions, categories, default='Moderate Water')

In [52]:
# Display the counts of each category
# Ensure the counts are correctly calculated
category_counts = drinking_waterpoints['category'].value_counts()
category_counts

category
Limited Water     151
Moderate Water     55
Optimal Water      42
Name: count, dtype: int64

In [53]:
drinking_waterpoints

,Id,ImageUrls,ImageUrls.,Status,Status.key,Submission,UserEmail,UserEmail.,_id,_ignored,...,index_right,id,rowid,latitude,lat_min,lat_max,longitude,lon_min,lon_max,category
0,"1,615",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:02:51.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,SHQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,197419,147427609.0,197420,6.501777,6.501371,6.502182,3.394455,3.393952,3.394957,Moderate Water
2,"1,605",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 11:21:21.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,RnQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,196869,147410640.0,196870,6.499344,6.498939,6.499750,3.393443,3.392940,3.393946,Limited Water
3,"1,617",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:07:24.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,RHQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,196865,147410636.0,196866,6.502587,6.502182,6.502993,3.393457,3.392954,3.393959,Moderate Water
4,"1,607",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 11:31:18.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,THQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,197977,147444585.0,197978,6.498534,6.498128,6.498939,3.395443,3.394941,3.395946,Limited Water
5,"1,612",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 11:55:40.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,S3Q0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,197976,147444584.0,197977,6.499344,6.498939,6.499750,3.395447,3.394944,3.395949,Limited Water
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279,"1,594",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 10:49:33.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,K3Q0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,195760,147376693.0,195761,6.501777,6.501371,6.502182,3.391450,3.390947,3.391952,Limited Water
280,"1,624",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:51:47.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,MnQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,196310,147393662.0,196311,6.504209,6.503804,6.504614,3.392462,3.391959,3.392964,Limited Water
281,"1,596",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 10:58:00.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,MXQ0CJABf4CpNA64g0Fo,ImageUrls.keyword,...,196314,147393666.0,196315,6.500966,6.500561,6.501371,3.392448,3.391945,3.392951,Limited Water
282,"1,623",https://fotoquest.blob.core.windows.net/fields...,https://fotoquest.blob.core.windows.net/fields...,Completed,Completed,"Oct 25, 2023 @ 01:39:30.000",umihechiluru@yahoo.com,umihechiluru@yahoo.com,MHQ0CJABf4CpNA64g0Fo,-,...,195758,147376691.0,195759,6.503398,6.502993,6.503804,3.391456,3.390954,3.391959,Limited Water


In [54]:
drinking_waterpoints.columns

Index(['Id', 'ImageUrls', 'ImageUrls.', 'Status', 'Status.key', 'Submission',
       'UserEmail', 'UserEmail.', '_id', '_ignored', '_index', '_score',
       'appVersion', 'appVersi_1', 'comment', 'comment.ke', 'dateSurvey',
       'dateSurv_1', 'doYouQueue', 'doYouQue_1', 'operatingS', 'operatin_1',
       'peopleTrav', 'peopleTr_1', 'responsibl', 'responsi_1', 'timeNeeded',
       'timeNeed_1', 'userPositi', 'userPosi_1', 'userPosi_2', 'userPosi_3',
       'userPosi_4', 'userPosi_5', 'userPosi_6', 'userPosi_7', 'volunteerL',
       'voluntee_1', 'waterDrink', 'waterDri_1', 'waterFree', 'waterFree.',
       'waterPubli', 'waterPub_1', 'waterSourc', 'waterSou_1', 'waterSou_2',
       'waterSou_3', 'waterSou_4', 'waterSou_5', 'waterState', 'waterSta_1',
       'waterTreat', 'waterTre_1', 'waterType', 'waterType.', 'waterTypeO',
       'waterTyp_1', 'waterUsed', 'waterUsed.', 'whySourceA', 'whySourc_1',
       'whySourc_2', 'whySourc_3', 'field_66', 'field_67', 'globalid',
       'uniq_i

In [55]:
# Assign a unique identifier to each water point
# Ensure the identifier is unique and sequential
drinking_waterpoints['waterpoint_id'] = range (1, len(drinking_waterpoints) + 1)

In [56]:
drinking_waterpoints = drinking_waterpoints[['Submission', 'doYouQueue', 'peopleTrav', 'timeNeeded', 'userPosi_1', 
                                             'userPosi_2', 'waterDrink', 'waterpoint_id', 'geometry', 
                                             'waterSourc', 'waterState', 'waterDrink_estimated', 'watertype_revised', 
                                             'waterPubli', 'waterFree', 'category', 'latitude', 
                                             'longitude', 'lon_min', 'lat_min', 'lon_max', 'lat_max']]
drinking_waterpoints

,Submission,doYouQueue,peopleTrav,timeNeeded,userPosi_1,userPosi_2,waterDrink,waterpoint_id,geometry,waterSourc,...,watertype_revised,waterPubli,waterFree,category,latitude,longitude,lon_min,lat_min,lon_max,lat_max
0,"Oct 25, 2023 @ 01:02:51.000","No, there's usually no queue",Within 1-5 mins of walking,54.18533,6.501800,3.394038,Don't know,1,POINT (3.3942 6.50181),Partially Functional,...,Unimproved,Privately Owned,Free,Moderate Water,6.501777,3.394455,3.393952,6.501371,3.394957,6.502182
2,"Oct 25, 2023 @ 11:21:21.000","No, there's usually no queue",Within 1-5 mins of walking,50.7915,6.499572,3.394045,No,2,POINT (3.39384 6.49949),Fully Functional,...,Unimproved,Privately Owned,Free,Limited Water,6.499344,3.393443,3.392940,6.498939,3.393946,6.499750
3,"Oct 25, 2023 @ 01:07:24.000","No, there's usually no queue",Within 1-5 mins of walking,40.289,6.502545,3.393935,Yes,3,POINT (3.39375 6.50247),Fully Functional,...,Unimproved,Privately Owned,Free,Moderate Water,6.502587,3.393457,3.392954,6.502182,3.393959,6.502993
4,"Oct 25, 2023 @ 11:31:18.000","No, there's usually no queue",Within 1-5 mins of walking,80.1333,6.498633,3.395297,No,4,POINT (3.39521 6.49856),Partially Functional,...,Unimproved,Privately Owned,Free,Limited Water,6.498534,3.395443,3.394941,6.498128,3.395946,6.498939
5,"Oct 25, 2023 @ 11:55:40.000","No, there's usually no queue",Within 1-5 mins of walking,158.109,6.498722,3.394865,No,5,POINT (3.39507 6.49923),Fully Functional,...,Unimproved,Publicly Owned,Non Free,Limited Water,6.499344,3.395447,3.394944,6.498939,3.395949,6.499750
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279,"Oct 25, 2023 @ 10:49:33.000",Sometimes it depends on the time of day\t,Within 1-5 mins of walking,62.6106,6.501483,3.391277,No,244,POINT (3.39158 6.50154),Fully Functional,...,Unimproved,Publicly Owned,Non Free,Limited Water,6.501777,3.391450,3.390947,6.501371,3.391952,6.502182
280,"Oct 25, 2023 @ 01:51:47.000",Sometimes it depends on the time of day\t,Within 1-5 mins of walking,49.78436,6.504860,3.392330,No,245,POINT (3.39231 6.50446),Fully Functional,...,Improved,Privately Owned,Free,Limited Water,6.504209,3.392462,3.391959,6.503804,3.392964,6.504614
281,"Oct 25, 2023 @ 10:58:00.000",Sometimes it depends on the time of day\t,Within 1-5 mins of walking,88.07617,6.500620,3.392213,No,246,POINT (3.39204 6.50114),Fully Functional,...,Unimproved,Privately Owned,Free,Limited Water,6.500966,3.392448,3.391945,6.500561,3.392951,6.501371
282,"Oct 25, 2023 @ 01:39:30.000",Sometimes it depends on the time of day\t,Within 1-5 mins of walking,88.61507,6.503625,3.392047,No,247,POINT (3.39194 6.50362),Fully Functional,...,Improved,Publicly Owned,Non Free,Limited Water,6.503398,3.391456,3.390954,6.502993,3.391959,6.503804


In [57]:
# Write to GeoJSON
drinking_waterpoints.to_file(data_inputs + 'Lagos_DW_Categorized.geojson', driver='GeoJSON')

In [3]:
drinking_waterpoints = gpd.read_file(data_inputs + 'Lagos_DW_V1.geojson')

# Population Grid Data

In [4]:
study_area = gpd.read_file(data_inputs + "100mGrid.gpkg")
population_data = data_inputs + "nga_ppp_2020_UNadj.tif"

In [5]:
study_area["grid_id"] = range(1, len(study_area) + 1)

In [6]:
with rasterio.open(population_data) as dataset:
    geometries = [study_area.geometry.unary_union.__geo_interface__]
    clipped_image, clipped_transform = mask(dataset, geometries, crop=True)
    band1 = clipped_image[0] # Read the first band of the raster

/var/folders/y0/vlr9p1691nzbmnxt09fwc1nw0000gp/T/ipykernel_15687/1658636788.py:2: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geometries = [study_area.geometry.unary_union.__geo_interface__]


In [7]:
out_meta = dataset.meta.copy()
out_meta.update({
        "height": clipped_image.shape[1],
        "width": clipped_image.shape[2],
        "transform": clipped_transform
    })

In [8]:
with rasterio.open(data_inputs + 'lagos_nga_ppp_2020_UNadj.tif', "w", **out_meta) as dest:
    dest.write(clipped_image)

In [9]:
# 1. Open the population raster
with rasterio.open(data_inputs + "lagos_nga_ppp_2020_UNadj.tif") as src:
    data = src.read(1)
    transform = src.transform
    crs = src.crs

    rows, cols = data.shape

    polygons = []
    values = []

    for row in range(rows):
        for col in range(cols):
            value = data[row, col]
            if value == src.nodata:
                continue  

            x, y = rasterio.transform.xy(transform, row, col, offset='ul')

            pixel_width = transform.a
            pixel_height = -transform.e 

            poly = box(x, y - pixel_height, x + pixel_width, y)
            polygons.append(poly)
            values.append(value)

population_lagos = gpd.GeoDataFrame({
    "value": values,
    "geometry": polygons
}, crs=crs)

# 7. Save to GeoPackage (two layers: polygons and centroids)
population_lagos.to_file(data_temp + "population_lagos.gpkg", driver="GPKG")

In [10]:
# 5. Calculate centroids
population_lagos["centroid"] = population_lagos.geometry.centroid

# 6. Create centroid layer (points)
population_centroids = population_lagos[["value", "centroid"]].copy()
population_centroids = population_centroids.rename(columns={"centroid": "geometry"})
population_centroids = population_centroids.set_geometry("geometry")
population_centroids.set_crs("EPSG:4326", inplace=True)

# 4. Add unique ID
population_centroids["id"] = range(1, len(population_centroids) + 1)

population_centroids.to_file(data_temp + "population_centroids_lagos.gpkg", driver="GPKG")

/var/folders/y0/vlr9p1691nzbmnxt09fwc1nw0000gp/T/ipykernel_15687/2587920847.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  population_lagos["centroid"] = population_lagos.geometry.centroid


In [11]:

population_centroids

,value,geometry,id
0,0.574855,POINT (3.21417 7.00333),1
1,0.555777,POINT (3.215 7.00333),2
2,0.581640,POINT (3.21583 7.00333),3
3,0.687089,POINT (3.21667 7.00333),4
4,0.785828,POINT (3.2175 7.00333),5
...,...,...,...
335258,7.163505,POINT (2.8325 6.39083),335259
335259,7.007490,POINT (2.83333 6.39083),335260
335260,6.619279,POINT (2.83417 6.39083),335261
335261,6.599503,POINT (2.835 6.39083),335262


In [12]:
# === Ensure same CRS ===
if study_area.crs != population_centroids.crs:
    population_centroids = population_centroids.to_crs(study_area.crs)

# === STEP 1: Spatial join to assign population values ===
join_gdf = gpd.sjoin(
    population_centroids,
    study_area,
    how="left",
    predicate="within"
)

# Compute mean population for each grid_id (works for 1 or 2 centroids)
pop_by_grid = (
    join_gdf.groupby("grid_id")
    .agg({"value": "mean"})
    .reset_index()
)

# Merge into grid
lagos_grid_with_pop = study_area.merge(pop_by_grid, on="grid_id", how="left")

# 5. Identify empty grids (no population assigned)
empty_grids = lagos_grid_with_pop[lagos_grid_with_pop['value'].isna()].copy()

if not empty_grids.empty:
    # 6. Build KDTree from population centroid coordinates
    pop_points = np.array([(pt.x, pt.y) for pt in population_centroids.geometry])
    pop_tree = cKDTree(pop_points)

    # 7. Get centroids of empty grids
    empty_centroids = np.array([(geom.centroid.x, geom.centroid.y) for geom in empty_grids.geometry])

    # 8. Query nearest population centroid for each empty grid centroid
    dist, idx = pop_tree.query(empty_centroids, k=1)

    # 9. Retrieve population values of nearest centroids
    nearest_values = population_centroids.iloc[idx]['value'].values

    # 10. Assign these values to the empty grids
    empty_grids['value'] = nearest_values
    lagos_grid_with_pop.loc[empty_grids.index, 'value'] = empty_grids['value']

# 11. Save the updated grid with population values to a GeoPackage
output_path = data_temp + "lagos_grid_pop.gpkg"
lagos_grid_with_pop.to_file(output_path, driver="GPKG")


In [13]:
lagos_grid_pop_centroids = lagos_grid_with_pop.copy()
lagos_grid_pop_centroids['centroid'] = lagos_grid_pop_centroids.geometry.centroid
lagos_grid_pop_centroids = lagos_grid_pop_centroids.set_geometry('centroid')
lagos_grid_pop_centroids = lagos_grid_pop_centroids.drop(columns='geometry')

/var/folders/y0/vlr9p1691nzbmnxt09fwc1nw0000gp/T/ipykernel_15687/1584816035.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  lagos_grid_pop_centroids['centroid'] = lagos_grid_pop_centroids.geometry.centroid


In [14]:
lagos_grid_pop_centroids.to_file(
    data_temp + 'lagos_grid_pop_centroids.geojson',
    driver='GeoJSON'
)

# OD matrix

In [15]:
# Read the Kano OD matrix
# Ensure the file path is correct and the file exists
Lagos_OD_Matrix = pd.read_csv(data_inputs + 'lagos_OD_Matrix.csv')
Lagos_OD_Matrix

,origin_id,destination_id,duration_seconds,distance_km,duration_min
0,34728,123,11249.8,15.6248,187.5
1,34728,122,23889.1,37.7881,398.2
2,34728,121,23854.7,37.7403,397.6
3,34728,124,28093.4,39.0189,468.2
4,34728,200,27989.7,38.8748,466.5
...,...,...,...,...,...
350706,130960,200,4593.3,6.3796,76.6
350707,130960,219,4653.7,6.4636,77.6
350708,130960,198,4605.0,6.3959,76.8
350709,130960,124,4697.0,6.5237,78.3


In [16]:
# Filter rows where duration_seconds <= 3600
Lagos_OD_Matrix_quota = Lagos_OD_Matrix[Lagos_OD_Matrix['duration_seconds'] <= 3600]  # 1 hour

Lagos_OD_Matrix_quota

,origin_id,destination_id,duration_seconds,distance_km,duration_min
815,131089,122,3573.9,4.9638,59.6
816,131089,121,3539.5,4.9160,59.0
855,131097,122,3259.1,4.5266,54.3
856,131097,121,3224.7,4.4788,53.7
860,131098,122,3172.6,4.4064,52.9
...,...,...,...,...,...
350491,130917,221,3500.0,4.8611,58.3
350496,130918,221,3523.3,4.8935,58.7
350501,130919,221,3542.5,4.9202,59.0
350506,130920,221,3520.1,4.8891,58.7


In [17]:
# Merge the Kano OD matrix with the population centroids grid cells
pop_centroid_matrix_quota = pd.merge(Lagos_OD_Matrix_quota, lagos_grid_with_pop[['grid_id', 'longitude', 'latitude', 'lon_min', 'lat_min', 'lon_max', 'lat_max','value', 'geometry']],
                             left_on='origin_id', right_on='grid_id', how='left')

In [18]:
# Display the merged population centroid matrix
# Ensure the data is correctly merged and displayed
pop_centroid_matrix_quota

,origin_id,destination_id,duration_seconds,distance_km,duration_min,grid_id,longitude,latitude,lon_min,lat_min,lon_max,lat_max,value,geometry
0,131089,122,3573.9,4.9638,59.6,131089,3.262071,6.463671,3.261568,6.463266,3.262573,6.464076,27.613865,"POLYGON ((3.26157 6.46408, 3.26257 6.46408, 3...."
1,131089,121,3539.5,4.9160,59.0,131089,3.262071,6.463671,3.261568,6.463266,3.262573,6.464076,27.613865,"POLYGON ((3.26157 6.46408, 3.26257 6.46408, 3...."
2,131097,122,3259.1,4.5266,54.3,131097,3.262045,6.457185,3.261543,6.456780,3.262547,6.457591,43.686481,"POLYGON ((3.26155 6.45759, 3.26255 6.45759, 3...."
3,131097,121,3224.7,4.4788,53.7,131097,3.262045,6.457185,3.261543,6.456780,3.262547,6.457591,43.686481,"POLYGON ((3.26155 6.45759, 3.26255 6.45759, 3...."
4,131098,122,3172.6,4.4064,52.9,131098,3.262042,6.456374,3.261539,6.455969,3.262544,6.456780,45.253689,"POLYGON ((3.26154 6.45678, 3.26254 6.45678, 3...."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98581,130917,221,3500.0,4.8611,58.3,130917,3.262632,6.603127,3.262129,6.602721,3.263135,6.603532,158.319489,"POLYGON ((3.26213 6.60353, 3.26313 6.60353, 3...."
98582,130918,221,3523.3,4.8935,58.7,130918,3.262629,6.602316,3.262126,6.601911,3.263131,6.602721,157.096603,"POLYGON ((3.26213 6.60272, 3.26313 6.60272, 3...."
98583,130919,221,3542.5,4.9202,59.0,130919,3.262625,6.601505,3.262123,6.601100,3.263128,6.601911,146.198151,"POLYGON ((3.26213 6.60191, 3.26313 6.60191, 3...."
98584,130920,221,3520.1,4.8891,58.7,130920,3.262622,6.600694,3.262119,6.600289,3.263125,6.601100,117.070007,"POLYGON ((3.26212 6.6011, 3.26312 6.6011, 3.26..."


In [19]:
# Rename columns for clarity
pop_centroid_matrix_quota = pop_centroid_matrix_quota.rename(columns={
    "longitude": "origin_lon",
    "latitude": "origin_lat",
    "lon_min": "origin_lon_min",
    "lat_min": "origin_lat_min",
    "lon_max": "origin_lon_max",
    "lat_max": "origin_lat_max",
    "destination_id": "dwp_id",
    "value": "population"
})
columns_to_keep = ["grid_id", "origin_lon", "origin_lat", "origin_lon_min", "origin_lat_min", "origin_lon_max", "origin_lat_max", "population", "geometry", "dwp_id", "duration_seconds","duration_min", "distance_km"]
pop_centroid_matrix_quota = pop_centroid_matrix_quota[columns_to_keep]

In [20]:
pop_centroid_matrix_quota

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,geometry,dwp_id,duration_seconds,duration_min,distance_km
0,131089,3.262071,6.463671,3.261568,6.463266,3.262573,6.464076,27.613865,"POLYGON ((3.26157 6.46408, 3.26257 6.46408, 3....",122,3573.9,59.6,4.9638
1,131089,3.262071,6.463671,3.261568,6.463266,3.262573,6.464076,27.613865,"POLYGON ((3.26157 6.46408, 3.26257 6.46408, 3....",121,3539.5,59.0,4.9160
2,131097,3.262045,6.457185,3.261543,6.456780,3.262547,6.457591,43.686481,"POLYGON ((3.26155 6.45759, 3.26255 6.45759, 3....",122,3259.1,54.3,4.5266
3,131097,3.262045,6.457185,3.261543,6.456780,3.262547,6.457591,43.686481,"POLYGON ((3.26155 6.45759, 3.26255 6.45759, 3....",121,3224.7,53.7,4.4788
4,131098,3.262042,6.456374,3.261539,6.455969,3.262544,6.456780,45.253689,"POLYGON ((3.26154 6.45678, 3.26254 6.45678, 3....",122,3172.6,52.9,4.4064
...,...,...,...,...,...,...,...,...,...,...,...,...,...
98581,130917,3.262632,6.603127,3.262129,6.602721,3.263135,6.603532,158.319489,"POLYGON ((3.26213 6.60353, 3.26313 6.60353, 3....",221,3500.0,58.3,4.8611
98582,130918,3.262629,6.602316,3.262126,6.601911,3.263131,6.602721,157.096603,"POLYGON ((3.26213 6.60272, 3.26313 6.60272, 3....",221,3523.3,58.7,4.8935
98583,130919,3.262625,6.601505,3.262123,6.601100,3.263128,6.601911,146.198151,"POLYGON ((3.26213 6.60191, 3.26313 6.60191, 3....",221,3542.5,59.0,4.9202
98584,130920,3.262622,6.600694,3.262119,6.600289,3.263125,6.601100,117.070007,"POLYGON ((3.26212 6.6011, 3.26312 6.6011, 3.26...",221,3520.1,58.7,4.8891


In [21]:
# Convert waterpoint_id to numeric and ensure it is an integer
drinking_waterpoints['waterpoint_id'] = pd.to_numeric(drinking_waterpoints['waterpoint_id'], errors='raise').astype(int)

print(drinking_waterpoints['waterpoint_id'].dtype) 

int64


In [22]:
print(pop_centroid_matrix_quota['dwp_id'].dtype) 

int64


In [23]:
# Merge the population centroid matrix with the drinking water points
# Ensure the merge is done correctly and the columns are aligned
# This will add the waterpoint_id and other relevant columns to the population centroid matrix
distances_duration_matrix = pd.merge(pop_centroid_matrix_quota, drinking_waterpoints[['waterpoint_id', 'category', 'userPosi_1', 'userPosi_2']], 
                     left_on='dwp_id', right_on='waterpoint_id', how='left')

In [24]:
distances_duration_matrix

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,geometry,dwp_id,duration_seconds,duration_min,distance_km,waterpoint_id,category,userPosi_1,userPosi_2
0,131089,3.262071,6.463671,3.261568,6.463266,3.262573,6.464076,27.613865,"POLYGON ((3.26157 6.46408, 3.26257 6.46408, 3....",122,3573.9,59.6,4.9638,122,Limited Water,6.455690,3.293241
1,131089,3.262071,6.463671,3.261568,6.463266,3.262573,6.464076,27.613865,"POLYGON ((3.26157 6.46408, 3.26257 6.46408, 3....",121,3539.5,59.0,4.9160,121,Limited Water,6.454200,3.292481
2,131097,3.262045,6.457185,3.261543,6.456780,3.262547,6.457591,43.686481,"POLYGON ((3.26155 6.45759, 3.26255 6.45759, 3....",122,3259.1,54.3,4.5266,122,Limited Water,6.455690,3.293241
3,131097,3.262045,6.457185,3.261543,6.456780,3.262547,6.457591,43.686481,"POLYGON ((3.26155 6.45759, 3.26255 6.45759, 3....",121,3224.7,53.7,4.4788,121,Limited Water,6.454200,3.292481
4,131098,3.262042,6.456374,3.261539,6.455969,3.262544,6.456780,45.253689,"POLYGON ((3.26154 6.45678, 3.26254 6.45678, 3....",122,3172.6,52.9,4.4064,122,Limited Water,6.455690,3.293241
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98581,130917,3.262632,6.603127,3.262129,6.602721,3.263135,6.603532,158.319489,"POLYGON ((3.26213 6.60353, 3.26313 6.60353, 3....",221,3500.0,58.3,4.8611,221,Moderate Water,6.620801,3.285185
98582,130918,3.262629,6.602316,3.262126,6.601911,3.263131,6.602721,157.096603,"POLYGON ((3.26213 6.60272, 3.26313 6.60272, 3....",221,3523.3,58.7,4.8935,221,Moderate Water,6.620801,3.285185
98583,130919,3.262625,6.601505,3.262123,6.601100,3.263128,6.601911,146.198151,"POLYGON ((3.26213 6.60191, 3.26313 6.60191, 3....",221,3542.5,59.0,4.9202,221,Moderate Water,6.620801,3.285185
98584,130920,3.262622,6.600694,3.262119,6.600289,3.263125,6.601100,117.070007,"POLYGON ((3.26212 6.6011, 3.26312 6.6011, 3.26...",221,3520.1,58.7,4.8891,221,Moderate Water,6.620801,3.285185


In [25]:
distances_duration_matrix = gpd.GeoDataFrame(distances_duration_matrix, geometry="geometry", crs="EPSG:4326")
distances_duration_matrix.to_file(data_temp + 'distances_duration_matrix_quota.geojson', driver='GeoJSON')

In [26]:
distances_duration_matrix

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,geometry,dwp_id,duration_seconds,duration_min,distance_km,waterpoint_id,category,userPosi_1,userPosi_2
0,131089,3.262071,6.463671,3.261568,6.463266,3.262573,6.464076,27.613865,"POLYGON ((3.26157 6.46408, 3.26257 6.46408, 3....",122,3573.9,59.6,4.9638,122,Limited Water,6.455690,3.293241
1,131089,3.262071,6.463671,3.261568,6.463266,3.262573,6.464076,27.613865,"POLYGON ((3.26157 6.46408, 3.26257 6.46408, 3....",121,3539.5,59.0,4.9160,121,Limited Water,6.454200,3.292481
2,131097,3.262045,6.457185,3.261543,6.456780,3.262547,6.457591,43.686481,"POLYGON ((3.26155 6.45759, 3.26255 6.45759, 3....",122,3259.1,54.3,4.5266,122,Limited Water,6.455690,3.293241
3,131097,3.262045,6.457185,3.261543,6.456780,3.262547,6.457591,43.686481,"POLYGON ((3.26155 6.45759, 3.26255 6.45759, 3....",121,3224.7,53.7,4.4788,121,Limited Water,6.454200,3.292481
4,131098,3.262042,6.456374,3.261539,6.455969,3.262544,6.456780,45.253689,"POLYGON ((3.26154 6.45678, 3.26254 6.45678, 3....",122,3172.6,52.9,4.4064,122,Limited Water,6.455690,3.293241
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98581,130917,3.262632,6.603127,3.262129,6.602721,3.263135,6.603532,158.319489,"POLYGON ((3.26213 6.60353, 3.26313 6.60353, 3....",221,3500.0,58.3,4.8611,221,Moderate Water,6.620801,3.285185
98582,130918,3.262629,6.602316,3.262126,6.601911,3.263131,6.602721,157.096603,"POLYGON ((3.26213 6.60272, 3.26313 6.60272, 3....",221,3523.3,58.7,4.8935,221,Moderate Water,6.620801,3.285185
98583,130919,3.262625,6.601505,3.262123,6.601100,3.263128,6.601911,146.198151,"POLYGON ((3.26213 6.60191, 3.26313 6.60191, 3....",221,3542.5,59.0,4.9202,221,Moderate Water,6.620801,3.285185
98584,130920,3.262622,6.600694,3.262119,6.600289,3.263125,6.601100,117.070007,"POLYGON ((3.26212 6.6011, 3.26312 6.6011, 3.26...",221,3520.1,58.7,4.8891,221,Moderate Water,6.620801,3.285185


In [27]:
# Count the number of water points in each category
# Ensure the counts are correctly calculated
category_counts = drinking_waterpoints['category'].value_counts()
print(category_counts)

category
Limited Water     151
Moderate Water     55
Optimal Water      42
Name: count, dtype: int64


In [28]:
# Display the unique categories in the distances_duration_matrix
print(distances_duration_matrix['category'].unique())

['Limited Water' 'Moderate Water' 'Optimal Water']


In [29]:
# Define the categories for water points
categories = {
    'Optimal_Water': ['Optimal Water'],
    'Moderate_Water': ['Moderate Water'],
    'Limited_Water': ['Limited Water']
} 

In [30]:
# step 1: Create subsets of the distances_duration_matrix based on the defined categories
# Ensure the subsets are correctly created and the data is filtered based on the categories
subsets = {
    key: distances_duration_matrix[
        distances_duration_matrix['category'].isin(values)
    ]
    for key, values in categories.items()
}

In [31]:
# Extract subsets for each category
Optimal_Water_subset = subsets['Optimal_Water']
Moderate_Water_subset = subsets['Moderate_Water']
Limited_Water_subset = subsets['Limited_Water']

In [32]:
# Step 2: Define a function to get the smallest duration_seconds per grid_id for each category
def get_closest(df, n=1):
    return df.groupby('grid_id').apply(lambda x: x.nsmallest(n, 'duration_min')).reset_index(drop=True)


In [33]:
# If the subsets are already created for each category, we apply the function to each subset:
Optimal_Water_closest = get_closest(Optimal_Water_subset)
Moderate_Water_closest = get_closest(Moderate_Water_subset)
Limited_Water_closest = get_closest(Limited_Water_subset)

/var/folders/y0/vlr9p1691nzbmnxt09fwc1nw0000gp/T/ipykernel_15687/1133250327.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('grid_id').apply(lambda x: x.nsmallest(n, 'duration_min')).reset_index(drop=True)
/var/folders/y0/vlr9p1691nzbmnxt09fwc1nw0000gp/T/ipykernel_15687/1133250327.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('grid_id').apply(lambda x: x.nsmalle

In [34]:
# Step 4: Concatenate the filtered results into a single DataFrame
distances_duration_matrix = pd.concat([
    Optimal_Water_closest, Moderate_Water_closest, Limited_Water_closest])

In [35]:
distances_duration_matrix

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,geometry,dwp_id,duration_seconds,duration_min,distance_km,waterpoint_id,category,userPosi_1,userPosi_2
0,136570,3.272601,6.590964,3.272099,6.590559,3.273104,6.591370,105.035088,"POLYGON ((3.2721 6.59137, 3.2731 6.59137, 3.27...",199,3588.1,59.8,4.9835,199,Optimal Water,6.583374,3.305262
1,136571,3.272598,6.590154,3.272095,6.589748,3.273100,6.590559,107.433891,"POLYGON ((3.2721 6.59056, 3.2731 6.59056, 3.27...",199,3592.6,59.9,4.9898,199,Optimal Water,6.583374,3.305262
2,136580,3.272568,6.582856,3.272066,6.582451,3.273071,6.583262,143.842224,"POLYGON ((3.27207 6.58326, 3.27307 6.58326, 3....",199,3552.0,59.2,4.9334,199,Optimal Water,6.583374,3.305262
3,137128,3.273606,6.591775,3.273104,6.591370,3.274109,6.592181,108.481865,"POLYGON ((3.27311 6.59218, 3.27411 6.59218, 3....",199,3501.9,58.4,4.8638,199,Optimal Water,6.583374,3.305262
4,137129,3.273603,6.590964,3.273100,6.590559,3.274106,6.591370,111.714867,"POLYGON ((3.2731 6.59137, 3.27411 6.59137, 3.2...",199,3511.8,58.5,4.8775,199,Optimal Water,6.583374,3.305262
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21160,314964,3.654756,6.466914,3.654253,6.466509,3.655259,6.467319,7.528485,"POLYGON ((3.65426 6.46732, 3.65526 6.46732, 3....",5,3550.3,59.2,4.9311,5,Limited Water,6.479160,3.770333
21161,315059,3.655765,6.468536,3.655262,6.468130,3.656267,6.468941,7.206224,"POLYGON ((3.65527 6.46894, 3.65627 6.46894, 3....",5,3419.7,57.0,4.7496,5,Limited Water,6.479160,3.770333
21162,315060,3.655761,6.467725,3.655259,6.467319,3.656264,6.468130,6.969397,"POLYGON ((3.65526 6.46813, 3.65626 6.46813, 3....",5,3475.7,57.9,4.8273,5,Limited Water,6.479160,3.770333
21163,315061,3.655758,6.466914,3.655255,6.466509,3.656260,6.467319,7.423367,"POLYGON ((3.65526 6.46732, 3.65626 6.46732, 3....",5,3565.2,59.4,4.9516,5,Limited Water,6.479160,3.770333


In [36]:
distances_duration_matrix = distances_duration_matrix.rename(columns={
    "userPosi_2": "dest_lon",
    "userPosi_1": "dest_lat"
})
distances_duration_matrix = distances_duration_matrix.drop(columns=['dwp_id'])

In [37]:
# Add geometry to the DataFrame because the OD matrix is a CSV file and does not have geometry information.
# We create a geometry column using the origin coordinates.
gdf = gpd.GeoDataFrame(distances_duration_matrix, geometry="geometry", crs="EPSG:4326")

In [38]:
# Ensure the geometry is correctly created from the origin coordinates
gpkg_path = data_temp + 'distances_duration_closest_DW.gpkg'
gdf.to_file(gpkg_path, layer="distances_duration_closest_DW", driver="GPKG")

In [39]:
# Review and remove
origin_dest = gpd.read_file(data_temp + "distances_duration_closest_DW.gpkg", layer="distances_duration_closest_DW")

In [40]:
origin_dest

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,duration_seconds,duration_min,distance_km,waterpoint_id,category,dest_lat,dest_lon,geometry
0,136570,3.272601,6.590964,3.272099,6.590559,3.273104,6.591370,105.035088,3588.1,59.8,4.9835,199,Optimal Water,6.583374,3.305262,"POLYGON ((3.2721 6.59137, 3.2731 6.59137, 3.27..."
1,136571,3.272598,6.590154,3.272095,6.589748,3.273100,6.590559,107.433891,3592.6,59.9,4.9898,199,Optimal Water,6.583374,3.305262,"POLYGON ((3.2721 6.59056, 3.2731 6.59056, 3.27..."
2,136580,3.272568,6.582856,3.272066,6.582451,3.273071,6.583262,143.842224,3552.0,59.2,4.9334,199,Optimal Water,6.583374,3.305262,"POLYGON ((3.27207 6.58326, 3.27307 6.58326, 3...."
3,137128,3.273606,6.591775,3.273104,6.591370,3.274109,6.592181,108.481865,3501.9,58.4,4.8638,199,Optimal Water,6.583374,3.305262,"POLYGON ((3.27311 6.59218, 3.27411 6.59218, 3...."
4,137129,3.273603,6.590964,3.273100,6.590559,3.274106,6.591370,111.714867,3511.8,58.5,4.8775,199,Optimal Water,6.583374,3.305262,"POLYGON ((3.2731 6.59137, 3.27411 6.59137, 3.2..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47337,314964,3.654756,6.466914,3.654253,6.466509,3.655259,6.467319,7.528485,3550.3,59.2,4.9311,5,Limited Water,6.479160,3.770333,"POLYGON ((3.65426 6.46732, 3.65526 6.46732, 3...."
47338,315059,3.655765,6.468536,3.655262,6.468130,3.656267,6.468941,7.206224,3419.7,57.0,4.7496,5,Limited Water,6.479160,3.770333,"POLYGON ((3.65527 6.46894, 3.65627 6.46894, 3...."
47339,315060,3.655761,6.467725,3.655259,6.467319,3.656264,6.468130,6.969397,3475.7,57.9,4.8273,5,Limited Water,6.479160,3.770333,"POLYGON ((3.65526 6.46813, 3.65626 6.46813, 3...."
47340,315061,3.655758,6.466914,3.655255,6.466509,3.656260,6.467319,7.423367,3565.2,59.4,4.9516,5,Limited Water,6.479160,3.770333,"POLYGON ((3.65526 6.46732, 3.65626 6.46732, 3...."


## Enhanced Two-Step Floating Catchment Area (E2SFCA) method

In [41]:
# Function to calculate beta based on the maximum distance and a given W value
from math import *
d = 60 * 60 # quota has been applied to limit the maximum duration to 1 hour
W = 0.01 # try 0.1, 0.05, 0.01, 0.75
beta = - d ** 2 / log(W)
print(beta)

2814228.242733072


In [42]:
# Display the first few rows of the origin_dest DataFrame
print(origin_dest.head())

   grid_id  origin_lon  origin_lat  origin_lon_min  origin_lat_min  \
0   136570    3.272601    6.590964        3.272099        6.590559   
1   136571    3.272598    6.590154        3.272095        6.589748   
2   136580    3.272568    6.582856        3.272066        6.582451   
3   137128    3.273606    6.591775        3.273104        6.591370   
4   137129    3.273603    6.590964        3.273100        6.590559   

   origin_lon_max  origin_lat_max  population  duration_seconds  duration_min  \
0        3.273104        6.591370  105.035088            3588.1          59.8   
1        3.273100        6.590559  107.433891            3592.6          59.9   
2        3.273071        6.583262  143.842224            3552.0          59.2   
3        3.274109        6.592181  108.481865            3501.9          58.4   
4        3.274106        6.591370  111.714867            3511.8          58.5   

   distance_km  waterpoint_id       category  dest_lat  dest_lon  \
0       4.9835          

In [43]:
# Convert 'duration' to numeric, coercing errors to NaN
origin_dest = origin_dest.copy()
origin_dest['duration_seconds'] = pd.to_numeric(origin_dest['duration_seconds'], errors='coerce')

In [44]:
# Drop rows with NaN values in 'duration' column
origin_dest = origin_dest.dropna(subset=['duration_seconds'])
origin_dest['grid_id'] = pd.to_numeric(origin_dest['grid_id'], errors='coerce')

In [45]:
origin_dest_acc = origin_dest

In [46]:
# Apply Gaussian decay function to calculate the weight of each grid to healthcare 
# facilities based on the travel duration. d is the travel time and beta is the decay 
# parameter previously calculated.
# The weight decreases as the duration increases, meaning facilities that are further away have less impact. 
origin_dest_acc['Weight'] = origin_dest_acc['duration_seconds'].apply(lambda d: round(math.exp(-d**2/beta), 8))

In [47]:
# Compute the Weighted Population (Pop_W), the population of each grid cell is multiplied 
# by the corresponding weight to calculate the weighted population.
origin_dest_acc['Pop_W'] = origin_dest_acc['population'] * origin_dest_acc['Weight']

In [48]:
origin_dest_acc

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,duration_seconds,duration_min,distance_km,waterpoint_id,category,dest_lat,dest_lon,geometry,Weight,Pop_W
0,136570,3.272601,6.590964,3.272099,6.590559,3.273104,6.591370,105.035088,3588.1,59.8,4.9835,199,Optimal Water,6.583374,3.305262,"POLYGON ((3.2721 6.59137, 3.2731 6.59137, 3.27...",0.010309,1.082767
1,136571,3.272598,6.590154,3.272095,6.589748,3.273100,6.590559,107.433891,3592.6,59.9,4.9898,199,Optimal Water,6.583374,3.305262,"POLYGON ((3.2721 6.59056, 3.2731 6.59056, 3.27...",0.010191,1.094851
2,136580,3.272568,6.582856,3.272066,6.582451,3.273071,6.583262,143.842224,3552.0,59.2,4.9334,199,Optimal Water,6.583374,3.305262,"POLYGON ((3.27207 6.58326, 3.27307 6.58326, 3....",0.011297,1.625040
3,137128,3.273606,6.591775,3.273104,6.591370,3.274109,6.592181,108.481865,3501.9,58.4,4.8638,199,Optimal Water,6.583374,3.305262,"POLYGON ((3.27311 6.59218, 3.27411 6.59218, 3....",0.012809,1.389543
4,137129,3.273603,6.590964,3.273100,6.590559,3.274106,6.591370,111.714867,3511.8,58.5,4.8775,199,Optimal Water,6.583374,3.305262,"POLYGON ((3.2731 6.59137, 3.27411 6.59137, 3.2...",0.012497,1.396081
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47337,314964,3.654756,6.466914,3.654253,6.466509,3.655259,6.467319,7.528485,3550.3,59.2,4.9311,5,Limited Water,6.479160,3.770333,"POLYGON ((3.65426 6.46732, 3.65526 6.46732, 3....",0.011346,0.085418
47338,315059,3.655765,6.468536,3.655262,6.468130,3.656267,6.468941,7.206224,3419.7,57.0,4.7496,5,Limited Water,6.479160,3.770333,"POLYGON ((3.65527 6.46894, 3.65627 6.46894, 3....",0.015679,0.112986
47339,315060,3.655761,6.467725,3.655259,6.467319,3.656264,6.468130,6.969397,3475.7,57.9,4.8273,5,Limited Water,6.479160,3.770333,"POLYGON ((3.65526 6.46813, 3.65626 6.46813, 3....",0.013669,0.095263
47340,315061,3.655758,6.466914,3.655255,6.466509,3.656260,6.467319,7.423367,3565.2,59.4,4.9516,5,Limited Water,6.479160,3.770333,"POLYGON ((3.65526 6.46732, 3.65626 6.46732, 3....",0.010926,0.081111


In [49]:
# Sum the Weighted Population
origin_dest_sum = origin_dest_acc.groupby(by='waterpoint_id')['Pop_W'].sum().reset_index()

In [50]:
origin_dest_sum

,waterpoint_id,Pop_W
0,1,29041.515897
1,2,13489.003912
2,3,930.403038
3,4,14406.140627
4,5,8348.220247
...,...,...
227,243,834.534982
228,244,676.227610
229,245,10858.790506
230,247,90814.886933


In [51]:
# Merge the Sum of Weighted Population Back into the Original Data
origin_dest_acc = origin_dest_acc.merge(origin_dest_sum, on='waterpoint_id')

In [52]:
origin_dest_acc

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,duration_seconds,duration_min,distance_km,waterpoint_id,category,dest_lat,dest_lon,geometry,Weight,Pop_W_x,Pop_W_y
0,136570,3.272601,6.590964,3.272099,6.590559,3.273104,6.591370,105.035088,3588.1,59.8,4.9835,199,Optimal Water,6.583374,3.305262,"POLYGON ((3.2721 6.59137, 3.2731 6.59137, 3.27...",0.010309,1.082767,26405.330433
1,136571,3.272598,6.590154,3.272095,6.589748,3.273100,6.590559,107.433891,3592.6,59.9,4.9898,199,Optimal Water,6.583374,3.305262,"POLYGON ((3.2721 6.59056, 3.2731 6.59056, 3.27...",0.010191,1.094851,26405.330433
2,136580,3.272568,6.582856,3.272066,6.582451,3.273071,6.583262,143.842224,3552.0,59.2,4.9334,199,Optimal Water,6.583374,3.305262,"POLYGON ((3.27207 6.58326, 3.27307 6.58326, 3....",0.011297,1.625040,26405.330433
3,137128,3.273606,6.591775,3.273104,6.591370,3.274109,6.592181,108.481865,3501.9,58.4,4.8638,199,Optimal Water,6.583374,3.305262,"POLYGON ((3.27311 6.59218, 3.27411 6.59218, 3....",0.012809,1.389543,26405.330433
4,137129,3.273603,6.590964,3.273100,6.590559,3.274106,6.591370,111.714867,3511.8,58.5,4.8775,199,Optimal Water,6.583374,3.305262,"POLYGON ((3.2731 6.59137, 3.27411 6.59137, 3.2...",0.012497,1.396081,26405.330433
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47337,314964,3.654756,6.466914,3.654253,6.466509,3.655259,6.467319,7.528485,3550.3,59.2,4.9311,5,Limited Water,6.479160,3.770333,"POLYGON ((3.65426 6.46732, 3.65526 6.46732, 3....",0.011346,0.085418,8348.220247
47338,315059,3.655765,6.468536,3.655262,6.468130,3.656267,6.468941,7.206224,3419.7,57.0,4.7496,5,Limited Water,6.479160,3.770333,"POLYGON ((3.65527 6.46894, 3.65627 6.46894, 3....",0.015679,0.112986,8348.220247
47339,315060,3.655761,6.467725,3.655259,6.467319,3.656264,6.468130,6.969397,3475.7,57.9,4.8273,5,Limited Water,6.479160,3.770333,"POLYGON ((3.65526 6.46813, 3.65626 6.46813, 3....",0.013669,0.095263,8348.220247
47340,315061,3.655758,6.466914,3.655255,6.466509,3.656260,6.467319,7.423367,3565.2,59.4,4.9516,5,Limited Water,6.479160,3.770333,"POLYGON ((3.65526 6.46732, 3.65626 6.46732, 3....",0.010926,0.081111,8348.220247


In [53]:
# supply value is set to 1 for simplicity (capacity of HCF)
# supply = 1
# in the future, we will link supply with ownership and EmOC service level
origin_dest_acc = origin_dest_acc.rename(columns={'Pop_W_y': 'Pop_W_S'})  # Pop_W_S: Population Weight Sum

In [54]:
supply_map = {
    'Optimal Water': 1.0,   
    'Moderate Water': 0.5,  
    'Limited Water': 0.2    
}

In [55]:
# Map the supply values to the categories
origin_dest_acc['supply'] = origin_dest_acc['category'].map(supply_map)

In [56]:
# Calculate the supply-demand ratio
# The supply-demand ratio is calculated by dividing the supply by the Pop_W_S (Population Weight Sum).
# This ratio indicates how well the supply meets the demand in each grid cell.
# A ratio greater than 1 indicates that the supply exceeds the demand, while a ratio less
origin_dest_acc['supply_demand_ratio'] = origin_dest_acc['supply'] / origin_dest_acc['Pop_W_S']
origin_dest_acc['supply_demand_ratio'].replace([np.inf, -np.inf, np.nan], 0, inplace=True)

/var/folders/y0/vlr9p1691nzbmnxt09fwc1nw0000gp/T/ipykernel_15687/2965326480.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  origin_dest_acc['supply_demand_ratio'].replace([np.inf, -np.inf, np.nan], 0, inplace=True)


In [57]:
# Calculate Rj * Weight for Each Grid Cell
origin_dest_acc['supply_W'] = origin_dest_acc['supply_demand_ratio'] * origin_dest_acc.Weight

In [58]:
# Compute Accessibility Index (Ai) for Each Grid Cell
origin_dest_acc['Accessibility'] = origin_dest_acc.groupby('grid_id')['supply_W'].transform('sum')

In [59]:
# Normalize
scaler = MinMaxScaler()
origin_dest_acc['Accessibility_standard'] = scaler.fit_transform(origin_dest_acc[['Accessibility']])

In [60]:
origin_dest_acc

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,population,duration_seconds,duration_min,...,dest_lon,geometry,Weight,Pop_W_x,Pop_W_S,supply,supply_demand_ratio,supply_W,Accessibility,Accessibility_standard
0,136570,3.272601,6.590964,3.272099,6.590559,3.273104,6.591370,105.035088,3588.1,59.8,...,3.305262,"POLYGON ((3.2721 6.59137, 3.2731 6.59137, 3.27...",0.010309,1.082767,26405.330433,1.0,0.000038,3.903992e-07,5.876121e-07,0.000087
1,136571,3.272598,6.590154,3.272095,6.589748,3.273100,6.590559,107.433891,3592.6,59.9,...,3.305262,"POLYGON ((3.2721 6.59056, 3.2731 6.59056, 3.27...",0.010191,1.094851,26405.330433,1.0,0.000038,3.859422e-07,5.809352e-07,0.000086
2,136580,3.272568,6.582856,3.272066,6.582451,3.273071,6.583262,143.842224,3552.0,59.2,...,3.305262,"POLYGON ((3.27207 6.58326, 3.27307 6.58326, 3....",0.011297,1.625040,26405.330433,1.0,0.000038,4.278447e-07,6.268104e-07,0.000093
3,137128,3.273606,6.591775,3.273104,6.591370,3.274109,6.592181,108.481865,3501.9,58.4,...,3.305262,"POLYGON ((3.27311 6.59218, 3.27411 6.59218, 3....",0.012809,1.389543,26405.330433,1.0,0.000038,4.850911e-07,7.281720e-07,0.000108
4,137129,3.273603,6.590964,3.273100,6.590559,3.274106,6.591370,111.714867,3511.8,58.5,...,3.305262,"POLYGON ((3.2731 6.59137, 3.27411 6.59137, 3.2...",0.012497,1.396081,26405.330433,1.0,0.000038,4.732688e-07,7.106955e-07,0.000105
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47337,314964,3.654756,6.466914,3.654253,6.466509,3.655259,6.467319,7.528485,3550.3,59.2,...,3.770333,"POLYGON ((3.65426 6.46732, 3.65526 6.46732, 3....",0.011346,0.085418,8348.220247,0.2,0.000024,2.718172e-07,2.718172e-07,0.000039
47338,315059,3.655765,6.468536,3.655262,6.468130,3.656267,6.468941,7.206224,3419.7,57.0,...,3.770333,"POLYGON ((3.65527 6.46894, 3.65627 6.46894, 3....",0.015679,0.112986,8348.220247,0.2,0.000024,3.756235e-07,3.756235e-07,0.000055
47339,315060,3.655761,6.467725,3.655259,6.467319,3.656264,6.468130,6.969397,3475.7,57.9,...,3.770333,"POLYGON ((3.65526 6.46813, 3.65626 6.46813, 3....",0.013669,0.095263,8348.220247,0.2,0.000024,3.274636e-07,3.274636e-07,0.000048
47340,315061,3.655758,6.466914,3.655255,6.466509,3.656260,6.467319,7.423367,3565.2,59.4,...,3.770333,"POLYGON ((3.65526 6.46732, 3.65626 6.46732, 3....",0.010926,0.081111,8348.220247,0.2,0.000024,2.617677e-07,2.617677e-07,0.000038


In [61]:
# Find the maximum value of the Accessibility_standard
max(origin_dest_acc.Accessibility_standard)

0.9999999999999999

In [62]:
# Create a GeoDataFrame from the origin_dest_acc DataFrame
# Ensure the geometry is correctly created from the origin coordinates
gdf = gpd.GeoDataFrame(origin_dest_acc, geometry='geometry', crs="EPSG:4326")
gpkg_path = data_temp + 'acc_score_closest.gpkg'
gdf.to_file(gpkg_path, layer="acc_score_closest", driver="GPKG")

## 4. Grouping by grid ID to prepare the final output file

In [63]:
# Read the GeoPackage file (if starting from this section)
results_grid = gpd.read_file(data_temp + 'acc_score_closest.gpkg')

In [64]:
results_grid = results_grid[['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'Accessibility_standard', 'geometry']]

In [65]:
# Remove duplicates based on the specified columns
# This ensures that each grid cell is unique in the results_grid DataFrame
results_grid = results_grid.drop_duplicates(['grid_id', 'origin_lon', 'origin_lat', 'origin_lon_min', 'origin_lat_min', 'origin_lon_max', 'origin_lat_max', 'Accessibility_standard', 'geometry'])

In [66]:
# save the results to a new GeoPackage file
output_gpkg_path = data_temp + 'drinking_water_deprivation_access.gpkg'
results_grid.to_file(output_gpkg_path, driver='GPKG')

In [67]:
results_grid

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,Accessibility_standard,geometry
0,136570,3.272601,6.590964,3.272099,6.590559,3.273104,6.591370,0.000087,"POLYGON ((3.2721 6.59137, 3.2731 6.59137, 3.27..."
1,136571,3.272598,6.590154,3.272095,6.589748,3.273100,6.590559,0.000086,"POLYGON ((3.2721 6.59056, 3.2731 6.59056, 3.27..."
2,136580,3.272568,6.582856,3.272066,6.582451,3.273071,6.583262,0.000093,"POLYGON ((3.27207 6.58326, 3.27307 6.58326, 3...."
3,137128,3.273606,6.591775,3.273104,6.591370,3.274109,6.592181,0.000108,"POLYGON ((3.27311 6.59218, 3.27411 6.59218, 3...."
4,137129,3.273603,6.590964,3.273100,6.590559,3.274106,6.591370,0.000105,"POLYGON ((3.2731 6.59137, 3.27411 6.59137, 3.2..."
...,...,...,...,...,...,...,...,...,...
47337,314964,3.654756,6.466914,3.654253,6.466509,3.655259,6.467319,0.000039,"POLYGON ((3.65426 6.46732, 3.65526 6.46732, 3...."
47338,315059,3.655765,6.468536,3.655262,6.468130,3.656267,6.468941,0.000055,"POLYGON ((3.65527 6.46894, 3.65627 6.46894, 3...."
47339,315060,3.655761,6.467725,3.655259,6.467319,3.656264,6.468130,0.000048,"POLYGON ((3.65526 6.46813, 3.65626 6.46813, 3...."
47340,315061,3.655758,6.466914,3.655255,6.466509,3.656260,6.467319,0.000038,"POLYGON ((3.65526 6.46732, 3.65626 6.46732, 3...."


### Setting values for Low medium and High categories

We started by defining equal value division, and modified the thesholds to a value that is more legible and easier to interpret. Every model should have their own thresholds based on the data distribution of the three categories. 

Note: For Kano, we excluded grid cells with index values below 0.000001 that indicated very low population and a small number of buildings.  


In [71]:
# Assign deprivation levels based on the Accessibility_standard values
results_grid['result'] = -1
results_grid.loc[results_grid['Accessibility_standard'] >= 0, 'result'] = 2 #(high deprivation)
results_grid.loc[results_grid['Accessibility_standard'] > 0.0008459838930005538, 'result'] = 1 #(medium deprivation)
results_grid.loc[results_grid['Accessibility_standard'] > 0.002074840534805958, 'result'] = 0 #(low deprivation)



### Setting values for focus areas

We defined the focus areas based on values for the different thresholds. We aim at participants helping us to confirm the selection of the city-specific thresholds.

In [72]:
# We set Ajegunle Ikorodu boundary as our focus area
# 1. Read community boundary (already EPSG:26391)
ajegunle_ikorodu = gpd.read_file(
    data_inputs + "Ajegunle_Ikorodu.gpkg",
    layer="ajegunle_ikorodu "
)

# 2. Create 10 km buffer (units = metres)
focus_area = (
    ajegunle_ikorodu
    .buffer(10000, join_style=2)
    .unary_union
)

# 3. Reproject grid to same CRS
results_grid_proj = results_grid.to_crs(ajegunle_ikorodu.crs)

# 4. Flag focused grid cells (100 m resolution)
results_grid_proj["focused"] = (
    results_grid_proj.intersects(focus_area)
    .astype(int)
)

# 5. Reproject back to EPSG:4326 for output/visualisation
results_grid = results_grid_proj.to_crs("EPSG:4326")


/var/folders/y0/vlr9p1691nzbmnxt09fwc1nw0000gp/T/ipykernel_15687/4212911230.py:12: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  .unary_union


In [73]:
results_grid

,grid_id,origin_lon,origin_lat,origin_lon_min,origin_lat_min,origin_lon_max,origin_lat_max,Accessibility_standard,geometry,focused,result
0,136570,3.272601,6.590964,3.272099,6.590559,3.273104,6.591370,0.000087,"POLYGON ((3.2721 6.59137, 3.2731 6.59137, 3.27...",0,2
1,136571,3.272598,6.590154,3.272095,6.589748,3.273100,6.590559,0.000086,"POLYGON ((3.2721 6.59056, 3.2731 6.59056, 3.27...",0,2
2,136580,3.272568,6.582856,3.272066,6.582451,3.273071,6.583262,0.000093,"POLYGON ((3.27207 6.58326, 3.27307 6.58326, 3....",0,2
3,137128,3.273606,6.591775,3.273104,6.591370,3.274109,6.592181,0.000108,"POLYGON ((3.27311 6.59218, 3.27411 6.59218, 3....",0,2
4,137129,3.273603,6.590964,3.273100,6.590559,3.274106,6.591370,0.000105,"POLYGON ((3.2731 6.59137, 3.27411 6.59137, 3.2...",0,2
...,...,...,...,...,...,...,...,...,...,...,...
47337,314964,3.654756,6.466914,3.654253,6.466509,3.655259,6.467319,0.000039,"POLYGON ((3.65426 6.46732, 3.65526 6.46732, 3....",0,2
47338,315059,3.655765,6.468536,3.655262,6.468130,3.656267,6.468941,0.000055,"POLYGON ((3.65527 6.46894, 3.65627 6.46894, 3....",0,2
47339,315060,3.655761,6.467725,3.655259,6.467319,3.656264,6.468130,0.000048,"POLYGON ((3.65526 6.46813, 3.65626 6.46813, 3....",0,2
47340,315061,3.655758,6.466914,3.655255,6.466509,3.656260,6.467319,0.000038,"POLYGON ((3.65526 6.46732, 3.65626 6.46732, 3....",0,2


In [74]:
# Remove rows where the result is -1 (no deprivation)
# This ensures that only relevant results are kept in the results_grid DataFrame
results_grid = results_grid.loc[results_grid['result'] != -1]

In [75]:
# Rename columns for clarity
# This makes the column names more descriptive and easier to understand
results_grid = results_grid.rename(columns={
    'origin_lon': 'longitude',
    'origin_lat': 'latitude',
    'origin_lon_min': 'lon_min',
    'origin_lat_min': 'lat_min',
    'origin_lon_max': 'lon_max',
    'origin_lat_max': 'lat_max'
})

In [76]:
# Remove the 'geometry' column if it is not needed
results_grid

,grid_id,longitude,latitude,lon_min,lat_min,lon_max,lat_max,Accessibility_standard,geometry,focused,result
0,136570,3.272601,6.590964,3.272099,6.590559,3.273104,6.591370,0.000087,"POLYGON ((3.2721 6.59137, 3.2731 6.59137, 3.27...",0,2
1,136571,3.272598,6.590154,3.272095,6.589748,3.273100,6.590559,0.000086,"POLYGON ((3.2721 6.59056, 3.2731 6.59056, 3.27...",0,2
2,136580,3.272568,6.582856,3.272066,6.582451,3.273071,6.583262,0.000093,"POLYGON ((3.27207 6.58326, 3.27307 6.58326, 3....",0,2
3,137128,3.273606,6.591775,3.273104,6.591370,3.274109,6.592181,0.000108,"POLYGON ((3.27311 6.59218, 3.27411 6.59218, 3....",0,2
4,137129,3.273603,6.590964,3.273100,6.590559,3.274106,6.591370,0.000105,"POLYGON ((3.2731 6.59137, 3.27411 6.59137, 3.2...",0,2
...,...,...,...,...,...,...,...,...,...,...,...
47337,314964,3.654756,6.466914,3.654253,6.466509,3.655259,6.467319,0.000039,"POLYGON ((3.65426 6.46732, 3.65526 6.46732, 3....",0,2
47338,315059,3.655765,6.468536,3.655262,6.468130,3.656267,6.468941,0.000055,"POLYGON ((3.65527 6.46894, 3.65627 6.46894, 3....",0,2
47339,315060,3.655761,6.467725,3.655259,6.467319,3.656264,6.468130,0.000048,"POLYGON ((3.65526 6.46813, 3.65626 6.46813, 3....",0,2
47340,315061,3.655758,6.466914,3.655255,6.466509,3.656260,6.467319,0.000038,"POLYGON ((3.65526 6.46732, 3.65626 6.46732, 3....",0,2


In [78]:
# Save the results to a new GeoPackage file
output_gpkg_path = model_outputs + 'drinking-water-access-classs-V2.gpkg'
results_grid.to_file(output_gpkg_path, layer='drinking-water-access-class-V2', driver='GPKG')

In [79]:
print(results_grid.columns)


Index(['grid_id', 'longitude', 'latitude', 'lon_min', 'lat_min', 'lon_max',
       'lat_max', 'Accessibility_standard', 'geometry', 'focused', 'result'],
      dtype='object')


In [80]:
# Save the results to a CSV file in the format required by the IDEAMAPS data ecosystem
results_table = results_grid.drop(columns=['Accessibility_standard', 'grid_id', 'geometry'])
results_table.to_csv(model_outputs + 'model-output.csv', index=False)

In [81]:
# Display the results table
results_table

,longitude,latitude,lon_min,lat_min,lon_max,lat_max,focused,result
0,3.272601,6.590964,3.272099,6.590559,3.273104,6.591370,0,2
1,3.272598,6.590154,3.272095,6.589748,3.273100,6.590559,0,2
2,3.272568,6.582856,3.272066,6.582451,3.273071,6.583262,0,2
3,3.273606,6.591775,3.273104,6.591370,3.274109,6.592181,0,2
4,3.273603,6.590964,3.273100,6.590559,3.274106,6.591370,0,2
...,...,...,...,...,...,...,...,...
47337,3.654756,6.466914,3.654253,6.466509,3.655259,6.467319,0,2
47338,3.655765,6.468536,3.655262,6.468130,3.656267,6.468941,0,2
47339,3.655761,6.467725,3.655259,6.467319,3.656264,6.468130,0,2
47340,3.655758,6.466914,3.655255,6.466509,3.656260,6.467319,0,2
